# 第 17 集：Pandas 合并——merge

    > 对应《Numpy & Pandas 数据处理教程》课程。Notebook 按“概念 → 示例 → 观察结果”的顺序整理，建议逐格运行，并尝试修改示例数据。

    ## 本节目标


- 掌握 inner、left、right 和 outer 连接
- 会按单键、多键和索引关联
- 识别重复键造成的行数膨胀


## 1. `merge` 是按键关联

`concat` 更像把表堆在一起；`merge` 更像数据库连接，根据共同键寻找对应记录。


In [ ]:
import pandas as pd

students = pd.DataFrame(
    {"学号": [1, 2, 3, 4], "姓名": ["小林", "小周", "小陈", "小吴"]}
)
scores = pd.DataFrame({"学号": [1, 2, 4, 5], "成绩": [88, 92, 85, 77]})
students, scores


## 2. `inner` 与 `left`

`inner` 只保留两边都匹配的键；`left` 保留左表全部记录，右表没有匹配时填入缺失值。


In [ ]:
inner = students.merge(scores, on="学号", how="inner")
left = students.merge(scores, on="学号", how="left")
inner, left


## 3. `outer` 与来源标记

`outer` 保留两边所有键。`indicator=True` 新增 `_merge` 列，显示每行来自左表、右表还是两边都有，特别适合检查漏匹配。


In [ ]:
outer = students.merge(scores, on="学号", how="outer", indicator=True)
outer


## 4. 多个键共同匹配

当一个字段不足以唯一确定记录时，把多个列名传给 `on`。只有所有键都匹配，记录才会连接。


In [ ]:
prices = pd.DataFrame(
    {"商品": ["苹果", "苹果", "香蕉"], "城市": ["北京", "上海", "北京"], "价格": [8, 9, 6]}
)
sales = pd.DataFrame(
    {"商品": ["苹果", "苹果", "香蕉"], "城市": ["北京", "广州", "北京"], "数量": [10, 7, 12]}
)

prices.merge(sales, on=["商品", "城市"], how="outer")


## 5. 左右键名不同

两张表的关联列名字不同时，使用 `left_on` 和 `right_on`。合并后可按需要删除重复含义的键列。


In [ ]:
users = pd.DataFrame({"用户编号": [1, 2], "姓名": ["小林", "小周"]})
orders = pd.DataFrame({"user_id": [1, 1, 2], "金额": [99, 35, 68]})

users.merge(orders, left_on="用户编号", right_on="user_id", how="left")


## 6. 同名非键列与后缀

如果两表有同名但不是连接键的列，使用 `suffixes` 标明来源，避免得到难懂的 `_x`、`_y`。


In [ ]:
left_table = pd.DataFrame({"编号": [1, 2], "状态": ["启用", "停用"]})
right_table = pd.DataFrame({"编号": [1, 2], "状态": ["已审核", "待审核"]})

left_table.merge(right_table, on="编号", suffixes=("_账户", "_审核"))


## 7. 用 `validate` 防止意外的多对多连接

重复键可能让结果行数成倍增长。知道键关系时，设置 `validate="one_to_one"`、`"one_to_many"` 或 `"many_to_one"`，不符合预期就立即报错。


In [ ]:
checked = students.merge(scores, on="学号", how="left", validate="one_to_one")
checked


## 本节小结

选择连接方式之前，先回答“以哪张表为主、哪些键必须保留”。合并后检查行数和 `_merge`，并尽量用 `validate` 约束键关系。
